In [1]:
import sys
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# audited splits: train is the 257-row cleaned set
train_df = pd.read_excel("../data_splits/train.xlsx")
val_df   = pd.read_excel("../data_splits/val.xlsx")
test_df  = pd.read_excel("../data_splits/test.xlsx")

labels = sorted(train_df["Label"].unique().tolist())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
for s in (train_df, val_df, test_df):
    s["label_id"] = s["Label"].map(label2id)

# RoBERTa tokenizes differently from DeBERTa — recompute truncation from raw Trace Content
def head_tail_truncate_ids(text, tokenizer, max_length=512):
    ids = tokenizer.encode(text, add_special_tokens=False)
    budget = max_length - 2
    if len(ids) <= budget:
        return text
    head = budget // 2
    tail = budget - head
    return tokenizer.decode(ids[:head] + ids[-tail:])

for split in (train_df, val_df, test_df):
    split["text_truncated"] = split["Trace Content"].apply(
        lambda t: head_tail_truncate_ids(t, tokenizer)
    )

print(f"\nTrain {len(train_df)} / Val {len(val_df)} / Test {len(test_df)}")
print(f"Label mapping: {label2id}")
print(f"Truncated in train: {(train_df['Trace Content'] != train_df['text_truncated']).sum()} / {len(train_df)}")

Python: 3.12.10
PyTorch: 2.6.0+cu124
CUDA available: True
Device: cuda


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

e:\Projects\agent-failure-detection\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\beeya\.cache\huggingface\hub\models--roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (579 > 512). Running this sequence through the model will result in indexing errors



Train 257 / Val 87 / Test 87
Label mapping: {'HALLUCINATION': 0, 'LOOP': 1, 'SUCCESS': 2, 'UNSAFE_EXECUTION': 3}
Truncated in train: 77 / 257


In [2]:
from datasets import Dataset

def to_hf(split_df):
    return Dataset.from_pandas(
        split_df[["text_truncated", "label_id"]].rename(
            columns={"text_truncated": "text", "label_id": "label"}
        )
    )

train_ds, val_ds, test_ds = to_hf(train_df), to_hf(val_df), to_hf(test_df)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512, padding=False)

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds   = val_ds.map(tokenize_fn, batched=True)
test_ds  = test_ds.map(tokenize_fn, batched=True)

print(train_ds)

Map:   0%|          | 0/257 [00:00<?, ? examples/s]

Map:   0%|          | 0/87 [00:00<?, ? examples/s]

Map:   0%|          | 0/87 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 257
})


In [3]:
from torch import nn
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding, EarlyStoppingCallback)
from sklearn.metrics import f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

num_labels = len(label2id)

class_weights = compute_class_weight(
    "balanced", classes=np.arange(num_labels), y=train_df["label_id"].values
)
class_weights_t = torch.tensor(class_weights, dtype=torch.float32).to("cuda")
print("Class weights:", dict(zip(label2id.keys(), class_weights.round(3))))

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = nn.CrossEntropyLoss(weight=class_weights_t)(
            outputs.logits.view(-1, num_labels), labels.view(-1)
        )
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels,
    id2label=id2label, label2id=label2id,
    dtype=torch.float32,
)

args = TrainingArguments(
    output_dir="../classifier/checkpoints_roberta",
    num_train_epochs=8,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_steps=26,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    save_total_limit=2,
    logging_steps=10,
    fp16=True,
    report_to="none",
    seed=42,
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()

Class weights: {'HALLUCINATION': np.float64(0.988), 'LOOP': np.float64(1.397), 'SUCCESS': np.float64(0.834), 'UNSAFE_EXECUTION': np.float64(0.931)}


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,1.352322,1.144230,0.517241,0.447382,0.469766
2,0.654749,0.748967,0.666667,0.680512,0.655467
3,0.421757,0.934409,0.655172,0.659727,0.647755
4,0.300657,0.735982,0.689655,0.717785,0.692493
5,0.263121,0.580108,0.701149,0.715500,0.702364
6,0.198309,0.926547,0.712644,0.724599,0.714279
7,0.140175,0.712883,0.701149,0.716859,0.701685
8,0.078925,0.818153,0.712644,0.727248,0.712835


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=264, training_loss=0.4568902671788678, metrics={'train_runtime': 1794.5508, 'train_samples_per_second': 1.146, 'train_steps_per_second': 0.147, 'total_flos': 537879572441328.0, 'train_loss': 0.4568902671788678, 'epoch': 8.0})

In [4]:
trainer.save_model("../classifier/final_model_roberta")
tokenizer.save_pretrained("../classifier/final_model_roberta")
print("saved")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved


In [5]:
from sklearn.metrics import classification_report, confusion_matrix

test_out = trainer.predict(test_ds)
test_preds = np.argmax(test_out.predictions, axis=-1)
test_labels = test_out.label_ids

print(classification_report(
    test_labels, test_preds,
    target_names=[id2label[i] for i in range(num_labels)],
    digits=3,
))
print("Confusion matrix (rows=true, cols=pred):")
print("Labels:", [id2label[i] for i in range(num_labels)])
print(confusion_matrix(test_labels, test_preds))

                  precision    recall  f1-score   support

   HALLUCINATION      0.867     0.591     0.703        22
            LOOP      0.941     1.000     0.970        16
         SUCCESS      0.535     0.885     0.667        26
UNSAFE_EXECUTION      1.000     0.522     0.686        23

        accuracy                          0.736        87
       macro avg      0.836     0.749     0.756        87
    weighted avg      0.816     0.736     0.737        87

Confusion matrix (rows=true, cols=pred):
Labels: ['HALLUCINATION', 'LOOP', 'SUCCESS', 'UNSAFE_EXECUTION']
[[13  0  9  0]
 [ 0 16  0  0]
 [ 2  1 23  0]
 [ 0  0 11 12]]


In [6]:
import pandas as pd
comparison = pd.DataFrame({
    "Class": ["HALLUCINATION", "LOOP", "SUCCESS", "UNSAFE_EXECUTION", "Macro F1"],
    "Majority": [0.000, 0.000, 0.460, 0.000, 0.115],
    "MiniLM+LR": [0.586, 0.353, 0.359, 0.558, 0.464],
    "TF-IDF+LR": [0.678, 0.476, 0.619, 0.885, 0.664],
    "RoBERTa-base": [0.703, 0.970, 0.667, 0.686, 0.756],
    "DeBERTa-v3": [0.696, 0.848, 0.735, 1.000, 0.820],
})
comparison.to_excel("../data_splits/model_comparison.xlsx", index=False)
print(comparison.to_string(index=False))

           Class  Majority  MiniLM+LR  TF-IDF+LR  RoBERTa-base  DeBERTa-v3
   HALLUCINATION     0.000      0.586      0.678         0.703       0.696
            LOOP     0.000      0.353      0.476         0.970       0.848
         SUCCESS     0.460      0.359      0.619         0.667       0.735
UNSAFE_EXECUTION     0.000      0.558      0.885         0.686       1.000
        Macro F1     0.115      0.464      0.664         0.756       0.820
